# Factory I/O — Lids & Bases 생산 공정 (Production Line) Modbus TCP 제어

Factory I/O 기본 장면 **Production Line** 의 두 머시닝 센터(**Lids center**, **Bases center**)를
Pymodbus 로 제어해서 Lid 와 Base 를 목표 수량만큼 생산하고 종료합니다.

| 항목 | 값 |
|---|---|
| 라이브러리 | `pymodbus` 3.10+ (`device_id=` 키워드 사용) |
| IP / Port | 이 PC: `192.168.0.16` / `502` (수업 PC: `210.119.14.56`) |
| device_id | `1` |
| 목표 수량 | 기본 10개 (변경 가능) |

**사용 순서**
1. Factory I/O 에서 *Production Line* 장면을 열고 `F4` → Driver = **Modbus TCP/IP Server** (기본 매핑 그대로 사용)
2. 시뮬레이션 **Play(▶, F5)** — 하단 상태가 *Modbus TCP/IP Server (Started)* 인지 확인
3. 아래 셀을 위에서부터 차례대로 실행. 수량은 6번 셀의 `TARGET_LIDS`, `TARGET_BASES` 로 변경

> 💡 Factory I/O 없이 시험하려면 같은 폴더의 `factoryio_sim.py` (가상 Factory I/O) 를 실행하고 `HOST = "127.0.0.1"` 로 바꾸세요.

In [ ]:
# ------------------------------------------------------------
# 0. 라이브러리 (최초 1회만 설치, 필요 시 주석 해제)
# ------------------------------------------------------------
# %pip install -U "pymodbus>=3.10"

import time
import pymodbus
from pymodbus.client import ModbusTcpClient

print("pymodbus version:", pymodbus.__version__)

## 1. 연결 설정

In [ ]:
# ------------------------------------------------------------
# 1. Modbus TCP 연결 설정
# ------------------------------------------------------------
HOST      = "192.168.0.16"    # Factory I/O 가 실행 중인 PC 의 IP (Driver 화면 상단에 표시됨)
# HOST    = "210.119.14.56"   # 수업 PC 로 접속할 때
PORT      = 502               # Modbus TCP 기본 포트
DEVICE_ID = 1                 # Pymodbus 3.10+ 의 device_id (구버전의 slave/unit 과 같은 의미)
TIMEOUT   = 3                 # 응답 대기 시간(초)

SCAN_TIME = 0.05              # 제어 루프 주기(초) - PLC 의 스캔 타임 역할

## 2. I/O 주소 맵 (Production Line 기본 매핑)

Factory I/O Driver 화면의 번호 = Modbus **0-based 주소** 입니다. (괄호 안은 Modicon 표기)

**Discrete Input (센서, FC02)** — `at entry / at exit` 센서는 **부품이 있으면 0 (Active-Low)**

| 주소 | Modicon | 태그 |
|---|---|---|
| 0 | 10001 | Lids at entry |
| 1 | 10002 | Lids center (busy) |
| 2 | 10003 | Lids center (has error) |
| 3 | 10004 | Lids at exit |
| 4 | 10005 | Bases at entry |
| 5 | 10006 | Bases center (busy) |
| 6 | 10007 | Bases center (has error) |
| 7 | 10008 | Bases at exit |
| 8 | 10009 | FACTORY I/O (Running) |

**Coil (액추에이터, FC15)**

| 주소 | Modicon | 태그 | 주소 | Modicon | 태그 |
|---|---|---|---|---|---|
| 0 | 00001 | Lids raw conveyor | 7 | 00008 | Bases raw conveyor |
| 1 | 00002 | Lids center (produce lids) | 8 | 00009 | Bases center (produce lids) |
| 2 | 00003 | Lids center (start) | 9 | 00010 | Bases center (start) |
| 3 | 00004 | Lids center (reset) | 10 | 00011 | Bases center (reset) |
| 4 | 00005 | Lids center (stop) | 11 | 00012 | Bases center (stop) |
| 5 | 00006 | Lids exit conveyor 1 | 12 | 00013 | Bases exit conveyor 1 |
| 6 | 00007 | Lids exit conveyor 2 | 13 | 00014 | Bases exit conveyor 2 |
| | | | 14 | 00015 | Exit conveyor (공용) |

**Holding Register (FC06)** : 0 (40001) = Lids counter, 1 (40002) = Bases counter

※ Lids/Bases emitter 와 Remover 는 장면에서 자동 동작하므로 매핑되어 있지 않습니다.

In [ ]:
# ------------------------------------------------------------
# 2. I/O 주소 맵 (Factory I/O Driver 화면 번호 = 0-based Modbus 주소)
#    매핑을 바꿨다면 이 셀의 숫자만 수정하면 됩니다.
# ------------------------------------------------------------

# ---- Discrete Inputs (센서) ----
DI = {
    "LID_ENTRY":   0,   # 10001  Lids at entry        (Active-Low: 0 = 부품 있음)
    "LID_BUSY":    1,   # 10002  Lids center (busy)
    "LID_ERROR":   2,   # 10003  Lids center (has error)
    "LID_EXIT":    3,   # 10004  Lids at exit         (Active-Low)
    "BASE_ENTRY":  4,   # 10005  Bases at entry       (Active-Low)
    "BASE_BUSY":   5,   # 10006  Bases center (busy)
    "BASE_ERROR":  6,   # 10007  Bases center (has error)
    "BASE_EXIT":   7,   # 10008  Bases at exit        (Active-Low)
    "FIO_RUNNING": 8,   # 10009  FACTORY I/O (Running) - 시뮬레이션 실행 중
}

# ---- Coils (액추에이터) ----
COIL = {
    "LID_RAW_CONV":   0,   # 00001  Lids raw conveyor (원자재 투입 컨베이어)
    "LID_PRODUCE":    1,   # 00002  Lids center (produce lids)  True = 뚜껑 생산
    "LID_START":      2,   # 00003  Lids center (start)  ※ ON 을 유지해야 가공이 진행됨
    "LID_RESET":      3,   # 00004  Lids center (reset)
    "LID_STOP":       4,   # 00005  Lids center (stop)   True = 일시정지
    "LID_EXIT_CONV1": 5,   # 00006  Lids exit conveyor 1
    "LID_EXIT_CONV2": 6,   # 00007  Lids exit conveyor 2
    "BASE_RAW_CONV":  7,   # 00008  Bases raw conveyor
    "BASE_PRODUCE":   8,   # 00009  Bases center (produce lids) False = 받침 생산
    "BASE_START":     9,   # 00010  Bases center (start)
    "BASE_RESET":    10,   # 00011  Bases center (reset)
    "BASE_STOP":     11,   # 00012  Bases center (stop)
    "BASE_EXIT_CONV1": 12, # 00013  Bases exit conveyor 1
    "BASE_EXIT_CONV2": 13, # 00014  Bases exit conveyor 2
    "EXIT_CONV":     14,   # 00015  Exit conveyor (Lid/Base 공용 출구, 끝에 Remover)
}

# ---- Holding Registers (디스플레이 카운터) ----
HR = {
    "LID_COUNTER":  0,  # 40001  Lids counter
    "BASE_COUNTER": 1,  # 40002  Bases counter
}

SENSOR_ACTIVE_LOW = True          # entry/exit 센서: 부품이 있으면 0

DI_COUNT   = max(DI.values()) + 1  # 한 번에 읽을 DI 개수
COIL_COUNT = max(COIL.values()) + 1

# ---- 라인별 구성 (Lid / Base 를 같은 코드로 제어하기 위한 표) ----
LINES = {
    "LID": dict(entry=DI["LID_ENTRY"], exit=DI["LID_EXIT"], busy=DI["LID_BUSY"], error=DI["LID_ERROR"],
                raw_conv=COIL["LID_RAW_CONV"], produce=COIL["LID_PRODUCE"], produce_lids=True,
                start=COIL["LID_START"], reset=COIL["LID_RESET"], stop=COIL["LID_STOP"],
                exit_conv=[COIL["LID_EXIT_CONV1"], COIL["LID_EXIT_CONV2"]],
                counter=HR["LID_COUNTER"]),
    "BASE": dict(entry=DI["BASE_ENTRY"], exit=DI["BASE_EXIT"], busy=DI["BASE_BUSY"], error=DI["BASE_ERROR"],
                 raw_conv=COIL["BASE_RAW_CONV"], produce=COIL["BASE_PRODUCE"], produce_lids=False,
                 start=COIL["BASE_START"], reset=COIL["BASE_RESET"], stop=COIL["BASE_STOP"],
                 exit_conv=[COIL["BASE_EXIT_CONV1"], COIL["BASE_EXIT_CONV2"]],
                 counter=HR["BASE_COUNTER"]),
}
print(f"DI {DI_COUNT}점 / Coil {COIL_COUNT}점 맵 로드 완료")

## 3. 연결 및 Modbus 헬퍼 함수

In [ ]:
# ------------------------------------------------------------
# 3. Modbus 클라이언트 생성 및 읽기/쓰기 헬퍼
# ------------------------------------------------------------
client = ModbusTcpClient(HOST, port=PORT, timeout=TIMEOUT)


def connect():
    """연결이 끊겨 있으면 다시 연결한다."""
    if not client.connected and not client.connect():
        raise ConnectionError(f"{HOST}:{PORT} 연결 실패 - Factory I/O Driver 가 Modbus TCP/IP Server 이고 Play 상태인지 확인")


def read_di():
    """Discrete Input 전체 읽기 (FC02) → bool 리스트."""
    rr = client.read_discrete_inputs(0, count=DI_COUNT, device_id=DEVICE_ID)
    if rr.isError():
        raise IOError(f"DI 읽기 실패: {rr}")
    return list(rr.bits[:DI_COUNT])      # bits 는 8의 배수로 패딩되므로 잘라서 사용


def part_present(di, addr):
    """entry/exit 센서 값 → 부품 감지 여부 (Active-Low 보정)."""
    return (not di[addr]) if SENSOR_ACTIVE_LOW else di[addr]


def write_all_coils(values):
    """Coil 0 ~ COIL_COUNT-1 을 한 번에 쓰기 (FC15) - 스캔당 1회 출력."""
    wr = client.write_coils(0, values, device_id=DEVICE_ID)
    if wr.isError():
        raise IOError(f"Coil 쓰기 실패: {wr}")


def write_hr(address, value):
    """Holding Register 1개 쓰기 (FC06) - 카운터 표시용."""
    wr = client.write_register(address, int(value), device_id=DEVICE_ID)
    if wr.isError():
        raise IOError(f"HR 쓰기 실패: {wr}")


def all_off():
    """모든 출력 OFF (종료/비상정지 시)."""
    try:
        connect()
        write_all_coils([False] * COIL_COUNT)
        print("모든 Coil OFF")
    except Exception as e:
        print("all_off 실패:", e)


# 연결 테스트 - 현재 센서 상태 출력
connect()
_di = read_di()
for name, addr in DI.items():
    print(f"  DI[{addr}] {name:12s} = {int(_di[addr])}")
if not _di[DI["FIO_RUNNING"]]:
    print("⚠️ Factory I/O 시뮬레이션이 정지 상태입니다. Play(▶) 를 눌러 주세요.")

## 4. 공정 순서 (라인별 상태머신)

Lid 라인과 Base 라인은 **같은 상태머신을 독립적으로 동시에** 실행합니다.

```
 ┌───────┐  생산수량 ≥ 목표  ┌───────┐  FLUSH_TIME 경과  ┌──────┐
 │  RUN  │ ───────────────▶ │ FLUSH │ ───────────────▶ │ DONE │
 └───────┘                  └───────┘                  └──────┘
```

1. **RUN** (생산)
   - `raw conveyor` ON : 이미터(자동)가 내보낸 원자재를 머시닝센터 입구로 이송 (끝단 스토퍼에 대기)
   - `produce lids` : Lid 라인 = True(뚜껑), Base 라인 = False(받침)
   - `start` **ON 유지** : 로봇이 입구의 원자재를 머시닝센터에 넣고 → 가공 → 출구 컨베이어에 배출 (start 를 끄면 일시정지)
   - `exit conveyor 1, 2` 와 공용 `Exit conveyor` ON : 완성품을 Remover 쪽으로 이송
   - `at exit` 센서에 부품이 **새로 감지될 때(엣지)** 생산수량 +1, Factory I/O 카운터(HR) 갱신
2. **FLUSH** : 목표 수량 도달 → `start`, `raw conveyor` OFF (추가 가공 중지), 출구 컨베이어만 잠시 더 돌려 완성품 배출
3. **DONE** : 해당 라인 정지. 두 라인이 모두 DONE 이면 모든 출력 OFF 후 종료

- `has error` 가 감지되면 `reset` 을 출력해 자동 복구 시도
- 공정 시작 시 `reset` 펄스로 머시닝센터 초기화

In [ ]:
# ------------------------------------------------------------
# 4. 라인 제어 상태머신
# ------------------------------------------------------------
FLUSH_TIME   = 4.0    # 목표 도달 후 출구 컨베이어를 더 돌리는 시간(초)
EXIT_DEBOUNCE = 0.3   # exit 센서 채터링 방지(초) - 이 시간 이상 감지되어야 1개로 인정


class LineController:
    """머시닝센터 1대(Lid 또는 Base)를 제어하는 상태머신."""

    def __init__(self, name, cfg, target):
        self.name, self.cfg, self.target = name, cfg, target
        self.produced = 0                 # 생산(배출) 완료 수량
        self.exit_counted = False         # 현재 exit 센서 위 부품을 이미 셌는지
        self.exit_since = None            # exit 센서 감지 시작 시각
        self.state = "RUN" if target > 0 else "DONE"
        self.t_state = time.time()

    def go(self, new_state):
        print(f"[{time.strftime('%H:%M:%S')}] {self.name:4s} {self.state} → {new_state} "
              f"(생산 {self.produced}/{self.target})")
        self.state, self.t_state = new_state, time.time()

    @property
    def done(self):
        return self.state == "DONE"

    def step(self, di, coils):
        """1 스캔: 입력(di) 을 보고 coils(출력 이미지) 에 값을 기록한다."""
        c = self.cfg
        now = time.time()

        # ---- 완성품 카운트: exit 센서가 EXIT_DEBOUNCE 이상 감지되면 1개 ----
        if part_present(di, c["exit"]):
            if self.exit_since is None:
                self.exit_since = now
            if not self.exit_counted and now - self.exit_since >= EXIT_DEBOUNCE:
                self.exit_counted = True
                if self.state != "DONE":
                    self.produced += 1
                    print(f"    ★ {self.name} 완성품: {self.produced}/{self.target}")
        else:
            self.exit_since, self.exit_counted = None, False

        if self.state == "DONE":
            return                                    # 출력 모두 OFF (기본값 유지)

        # ---- 공통: 출구 컨베이어 운전, 품목 선택 ----
        for a in c["exit_conv"]:
            coils[a] = True
        coils[c["produce"]] = c["produce_lids"]

        if self.state == "RUN":
            if di[c["error"]]:                        # 에러 → Reset
                coils[c["reset"]] = True
                return
            coils[c["raw_conv"]] = True               # 원자재 이송
            coils[c["start"]] = True                  # 가공 운전 (ON 유지)
            if self.produced >= self.target:
                self.go("FLUSH")

        elif self.state == "FLUSH":
            coils[c["start"]] = False                 # 추가 가공 중지
            if now - self.t_state >= FLUSH_TIME:
                self.go("DONE")

## 5. 메인 제어 루프

In [ ]:
# ------------------------------------------------------------
# 5. 메인 루프: 입력 읽기 → 로직 → 출력 쓰기 (PLC 스캔 방식)
# ------------------------------------------------------------
def reset_machines():
    """두 머시닝센터 Reset 펄스 (공정 시작 전 초기화)."""
    coils = [False] * COIL_COUNT
    for cfg in LINES.values():
        coils[cfg["reset"]] = True
    write_all_coils(coils)
    time.sleep(1.0)
    write_all_coils([False] * COIL_COUNT)
    time.sleep(0.5)


def run(target_lids=10, target_bases=10, print_every=5.0):
    """목표 수량만큼 Lid/Base 를 생산하고 종료한다."""
    connect()
    lines = [LineController("LID", LINES["LID"], target_lids),
             LineController("BASE", LINES["BASE"], target_bases)]
    print(f"=== 공정 시작: Lid {target_lids}개, Base {target_bases}개 ===")
    reset_machines()
    for l in lines:
        write_hr(l.cfg["counter"], 0)              # Factory I/O 카운터 0 으로

    t0 = time.time()
    t_print = time.time()
    shown = {}
    try:
        while not all(l.done for l in lines):
            di = read_di()                          # 1) 입력 스캔

            if not di[DI["FIO_RUNNING"]]:           # Factory I/O 가 Pause/Stop 이면 대기
                time.sleep(0.5)
                continue

            coils = [False] * COIL_COUNT            # 2) 로직 실행
            for l in lines:
                l.step(di, coils)
            coils[COIL["EXIT_CONV"]] = True         # 공용 출구 컨베이어 (Remover 쪽)

            write_all_coils(coils)                  # 3) 출력 갱신
            for l in lines:                         #    카운터가 바뀐 경우만 HR 쓰기
                if shown.get(l.name) != l.produced:
                    write_hr(l.cfg["counter"], l.produced)
                    shown[l.name] = l.produced

            if time.time() - t_print >= print_every:  # 4) 상태 출력
                print("    " + " | ".join(f"{l.name}:{l.state} {l.produced}/{l.target}" for l in lines))
                t_print = time.time()

            time.sleep(SCAN_TIME)

        print(f"=== 공정 완료: Lid {lines[0].produced}개, Base {lines[1].produced}개, "
              f"소요 {time.time() - t0:.1f}s ===")
    except KeyboardInterrupt:
        print("사용자 중지 (Kernel ▸ Interrupt)")
    except Exception as e:
        print("오류로 중지:", e)
    finally:
        all_off()                                   # 어떤 경우에도 출력 OFF
    return lines

## 6. 실행

목표 수량을 바꾸려면 아래 값만 수정하세요. 중간에 멈추려면 Jupyter 의 **■ (Interrupt kernel)** 버튼을 누르면 모든 출력이 OFF 됩니다.

In [ ]:
# ------------------------------------------------------------
# 6. 공정 실행 - 목표 수량 설정 (변경 가능)
# ------------------------------------------------------------
TARGET_LIDS  = 10   # Lid 목표 수량
TARGET_BASES = 10   # Base 목표 수량

result = run(target_lids=TARGET_LIDS, target_bases=TARGET_BASES)

In [ ]:
# ------------------------------------------------------------
# 7. 연결 종료
# ------------------------------------------------------------
all_off()
client.close()
print("Modbus 연결 종료")